# Vigil — Notebook 02: Compuerta de Calidad y Outliers

Este cuaderno implementa la Fase 2: Aplicación del protocolo de calidad, anonimización PII y detección de anomalías de engagement con Isolation Forest.

In [ ]:
import polars as pl
import yaml
import sys
sys.path.append('..')
from src.processing.quality_gate import ejecutar_compuerta_calidad, validar_esquema_silver

with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

## 1. Cargar Datos Silver Incompletos

In [ ]:
posts_silver = pl.read_parquet("../data/silver/redes_sociales/fb_posts_silver.parquet")
comments_silver = pl.read_parquet("../data/silver/redes_sociales/fb_comments_silver.parquet")

## 2. Ejecutar Compuerta de Calidad (Isolation Forest + PII Anonimización)

In [ ]:
# Ejecutar compuerta
posts_clean = ejecutar_compuerta_calidad(posts_silver, config["catalogo_seguidores"], "DIFNay")
comments_clean = ejecutar_compuerta_calidad(comments_silver, config["catalogo_seguidores"], "GeraldinePonceMexico")

print("Registros limpios - Posts:", posts_clean.height)
print("Registros limpios - Comentarios:", comments_clean.height)

## 3. Revisión de Anomalías de Engagement (Likes/Shares)

In [ ]:
anomalias_posts = posts_clean.filter(pl.col("is_anomaly") == True)
anomalias_comments = comments_clean.filter(pl.col("is_anomaly") == True)

print("Posts marcados como anomalías (Outliers de boosting):", anomalias_posts.height)
print("Comentarios marcados como anomalías:", anomalias_comments.height)

if anomalias_posts.height > 0:
    print("\nMuestras de posts anómalos:")
    print(anomalias_posts.select(["reacciones_totales", "tasa_interaccion", "texto_publicacion"]).head(5))

## 4. Validar Esquema Silver y Guardar Datos Validados

In [ ]:
assert validar_esquema_silver(posts_clean)
assert validar_esquema_silver(comments_clean)

posts_clean.write_parquet("../data/silver/redes_sociales/fb_posts_clean.parquet")
comments_clean.write_parquet("../data/silver/redes_sociales/fb_comments_clean.parquet")

print("Fase 2 Completada: Datasets validados y libres de anomalías listos para NLP.")